# Momentum One — the SELF-TUNER
### It doesn't just practice — it tunes its OWN rewards and settings toward staying consistent, and keeps a change only if it truly helps.

**FIRST: turn on the GPU.** Menu -> **Runtime** -> **Change runtime type** -> **L4 GPU** (or **A100** if you have it) -> **Save**.

## Cell 1 - your two numbers + load the bot (run once)
Set your **daily target %** and **daily risk %** at the top of the cell, then run it. A window pops up to connect your **Google Drive** (click Allow) - that's how the best brain and the safety anchor survive Colab shutting off.

In [ ]:
# ================== YOUR TWO NUMBERS (the ONLY things you set) ==================
TARGET_PERCENT = 3.0     # daily target: how much you want to make each day
RISK_PERCENT   = 3.5     # daily max loss: the most it may lose in a day (drawdown)
# ===============================================================================

import torch, os, shutil, re
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'NONE  <-  Runtime > Change runtime type > L4 (or A100), then re-run')

from google.colab import drive
drive.mount('/content/drive')

# get the code (must be pushed to GitHub first)
!git clone -q https://github.com/monty313/the-truth.git
%cd the-truth
!pip -q install pyyaml >/dev/null 2>&1

# write your two numbers into the one config the whole bot reads
src = open('configs/goals.yaml').read()
src = re.sub(r'(?m)^(goal_pct:\s*)[\d.]+',  r'\g<1>' + str(float(TARGET_PERCENT)), src, count=1)
src = re.sub(r'(?m)^(floor_pct:\s*)[\d.]+', r'\g<1>' + str(float(RISK_PERCENT)),  src, count=1)
open('configs/goals.yaml','w').write(src)
print(f'target {TARGET_PERCENT}% / risk {RISK_PERCENT}%  ->  written to configs/goals.yaml')

# the best brain + safety anchor survive Colab shutting off, via your Drive

# ---- Drive link (hardened, review 2026-07-20): safe to re-run; frozen brains always
# ---- reach Drive; known-bad (timid-era) brains are purged BY SERIAL NUMBER so training
# ---- never resumes a brain we've proven can't make money.
import hashlib
DR = '/content/drive/MyDrive/momentum_selftuner'
os.makedirs(DR, exist_ok=True)
KNOWN_BAD = ['745bafe170f2', '461dd4bf4dbd']   # sha256[:12] serials of the proven-flat era (records were counterfeit too)
def sn(p):
    return hashlib.sha256(open(p,'rb').read()).hexdigest()[:12]
for f in list(os.listdir(DR)):
    p = os.path.join(DR, f)
    if f.endswith('.pt') and os.path.isfile(p) and sn(p) in KNOWN_BAD:
        os.remove(p); print('purged known-flat brain from Drive:', f)
gp = os.path.join(DR, 'gpu_progress.json')      # its streak bar came from the counterfeit metric
if os.path.exists(gp) and not os.path.exists(gp + '.pre_fix_backup'):
    os.rename(gp, gp + '.pre_fix_backup'); print('old record book set aside (pre-fix metric)')
CK = 'artifacts/checkpoints'
if not os.path.islink(CK):                           # first run on this VM
    for f in os.listdir(CK):
        s2, dst = os.path.join(CK, f), os.path.join(DR, f)
        if f.endswith('.pt') and (not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(s2)):
            shutil.copy(s2, dst); print('-> Drive:', f)
    shutil.rmtree(CK, ignore_errors=True)
    os.symlink(DR, CK)
print('checkpoints live on your Drive:', DR)
print('ready.')


## Cell 2 - START (self-tune toward consistency)
Runs until Colab stops it. **Re-run any time to continue** from your best brain - it picks up at the exact generation.

What it does each round: tries a batch of small tweaks, trains each a little, and **keeps one only if it clearly clears more days - on fresh days, twice, and without dropping below its best.** The champion is never lost.

In [ ]:
# Runs until Colab stops it. Re-run any time to CONTINUE from your best brain.
# It self-tunes toward consistency and only KEEPS a change that truly helps (never a step back).
!python scripts/self_tune.py


## Cell 3 - how is it doing? (optional)

In [ ]:
import json
p = 'artifacts/checkpoints/meta_progress.json'
if os.path.exists(p):
    d = json.load(open(p))
    print(f"generation {d['gen']}  |  best streak: {d['best_streak']} cleared days in a row")
    print(f"held-out (audit) consistency: {d['audit_anchor_consistency']}")
    print(f"last round adopted a change: {d['last_adopt']}  |  exploring at: {d['explore']}")
else:
    print('no progress yet - let Cell 2 run a bit')
print('\n--- saved record brains (passNNNN = cleared days in a row) ---')
!ls -1 artifacts/checkpoints/history/ 2>/dev/null | grep meta_pass || echo 'no records yet'


## Cell 4 - get the best brain (optional)
Both the best brain and the safety anchor auto-save to **MyDrive / momentum_selftuner**. This also copies them to your computer.

In [ ]:
# best brain + the safety anchor also live in MyDrive/momentum_selftuner
from google.colab import files
for f in ('meta_best.pt','meta_anchor.pt'):
    pth = f'artifacts/checkpoints/{f}'
    if os.path.exists(pth): files.download(pth)
